In [1]:
!./myenv/bin/pip install faiss-cpu sentence-transformers numpy

In [3]:
import json
import pickle
import faiss
import numpy as np

from sentence_transformers import SentenceTransformer

/home/murgi/Documents/rag_gbm/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
with open("metrics.json", "r") as f:
    vision_data = json.load(f)

print(f"Loaded {len(vision_data)} BraTS cases")


Loaded 989 BraTS cases


In [5]:
def build_query(metrics):

    query = f"""
    Glioblastoma located in the {metrics['hemisphere']} hemisphere.

    Tumor located in the
    {metrics['coronal_location']}
    {metrics['axial_location']} region.

    Whole tumor volume:
    {metrics['wt_volume_cc']} cc.

    Tumor core volume:
    {metrics['tc_volume_cc']} cc.

    Enhancing tumor volume:
    {metrics['et_volume_cc']} cc.

    Necrotic core volume:
    {metrics['ncr_volume_cc']} cc.

    Edema volume:
    {metrics['ed_volume_cc']} cc.

    ET/TC ratio:
    {metrics['et_tc_ratio']}.

    NCR/TC ratio:
    {metrics['ncr_tc_ratio']}.

    Edema/WT ratio:
    {metrics['ed_wt_ratio']}.

    Solidity:
    {metrics['solidity']}.

    Elongation:
    {metrics['elongation']}.

    Sphericity:
    {metrics['sphericity']}.

    Enhancement ratio:
    {metrics.get('enhancement_ratio', 0)}.
    """

    return query


In [6]:
def interpret_metrics(metrics):

    findings = []

    if metrics["et_tc_ratio"] > 0.8:
        findings.append(
            "Predominantly enhancing tumor core"
        )

    if metrics["ncr_tc_ratio"] > 0.4:
        findings.append(
            "Significant necrotic component"
        )

    if metrics["ed_wt_ratio"] > 0.7:
        findings.append(
            "Extensive peritumoral edema"
        )

    if metrics.get("enhancement_ratio", 0) > 10:
        findings.append(
            "Marked contrast enhancement"
        )

    if metrics["sphericity"] > 0.8:
        findings.append(
            "Relatively regular tumor morphology"
        )

    if metrics["solidity"] < 0.5:
        findings.append(
            "Irregular infiltrative tumor pattern"
        )

    return findings


In [7]:
MODEL_NAME = "BAAI/bge-large-en-v1.5"

print("Loading encoder...")

encoder = SentenceTransformer(
    MODEL_NAME
)

print("Encoder loaded")

Loading encoder...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 656.08it/s]


Encoder loaded


In [8]:
print("Loading FAISS index...")

index = faiss.read_index(
    "cgga.index"
)

print("FAISS loaded")

Loading FAISS index...
FAISS loaded


In [9]:
with open(
        "documents.pkl",
        "rb"
) as f:

    documents = pickle.load(f)

with open(
        "metadata.pkl",
        "rb"
) as f:

    metadata = pickle.load(f)

print(
    f"Loaded {len(documents)} CGGA documents"
)


Loaded 693 CGGA documents


In [14]:
all_results = []

for case_num, case in enumerate(
        vision_data,
        start=1):

    try:

        case_id = case["case_id"]

        metrics = case["metrics"]

        # ===================================
        # SEMANTIC FINDINGS
        # ===================================

        semantic_findings = interpret_metrics(
            metrics
        )

        # Tumor burden category
        if metrics["wt_volume_cc"] > 40:

            semantic_findings.append(
                "Large tumor burden"
            )

        elif metrics["wt_volume_cc"] > 20:

            semantic_findings.append(
                "Moderate tumor burden"
            )

        else:

            semantic_findings.append(
                "Small tumor burden"
            )

        # ===================================
        # BUILD QUERY
        # ===================================

        query = build_query(
            metrics
        )

        query += "\n\nClinical interpretation:\n"

        for finding in semantic_findings:

            query += f"- {finding}\n"

        # ===================================
        # EMBEDDING
        # ===================================

        query_embedding = encoder.encode(
            query,
            normalize_embeddings=True
        )

        query_embedding = np.asarray(
            query_embedding,
            dtype=np.float32
        )

        # ===================================
        # SEARCH
        # ===================================

        scores, indices = index.search(
            query_embedding.reshape(1, -1),
            5
        )

        retrieved_patients = []

        idh_counts = {}
        mgmt_counts = {}

        os_values = []

        # ===================================
        # TOP 5 PATIENTS
        # ===================================

        for rank, idx in enumerate(
                indices[0],
                start=1):

            meta = metadata[idx]

            os_values.append(
                meta["os_days"]
            )

            retrieved_patients.append(
                {
                    "rank":
                        rank,

                    "similarity":
                        float(
                            scores[0][rank - 1]
                        ),

                    "confidence":
                        round(
                            float(
                                scores[0][rank - 1]
                            ) * 100,
                            2
                        ),

                    "metadata":
                        meta
                }
            )

            idh = meta.get(
                "idh",
                "Unknown"
            )

            mgmt = meta.get(
                "mgmt",
                "Unknown"
            )

            idh_counts[idh] = (
                idh_counts.get(idh, 0) + 1
            )

            mgmt_counts[mgmt] = (
                mgmt_counts.get(mgmt, 0) + 1
            )

        # ===================================
        # COHORT STATISTICS
        # ===================================

        avg_os = round(
            sum(os_values) / len(os_values),
            1
        )

        avg_similarity = round(
            np.mean(scores[0]),
            4
        )

        dominant_idh = max(
            idh_counts,
            key=idh_counts.get
        )

        dominant_mgmt = max(
            mgmt_counts,
            key=mgmt_counts.get
        )

        top_patient = (
            retrieved_patients[0]
            ["metadata"]
        )

        # ===================================
        # SAVE RESULT
        # ===================================

        all_results.append(
            {
                "case_id":
                    case_id,

                "vision_metrics":
                    metrics,

                "semantic_findings":
                    semantic_findings,

                "cohort_summary":
                    {
                        "idh_distribution":
                            idh_counts,

                        "mgmt_distribution":
                            mgmt_counts,

                        "dominant_idh":
                            dominant_idh,

                        "dominant_mgmt":
                            dominant_mgmt,

                        "average_os_days":
                            avg_os,

                        "average_similarity":
                            avg_similarity
                    },

                "top_match":
                    {
                        "patient_id":
                            top_patient[
                                "patient_id"
                            ],

                        "idh":
                            top_patient[
                                "idh"
                            ],

                        "mgmt":
                            top_patient[
                                "mgmt"
                            ],

                        "os_days":
                            top_patient[
                                "os_days"
                            ]
                    },

                "retrieved_patients":
                    retrieved_patients
            }
        )

        if case_num % 50 == 0:

            print(
                f"Processed "
                f"{case_num}/"
                f"{len(vision_data)}"
            )

    except Exception as e:

        print(
            f"Failed on "
            f"{case['case_id']}: "
            f"{e}"
        )

Processed 50/989
Processed 100/989
Processed 150/989
Processed 200/989
Processed 250/989
Processed 300/989
Processed 350/989
Processed 400/989
Processed 450/989
Processed 500/989
Processed 550/989
Processed 600/989
Processed 650/989
Processed 700/989
Processed 750/989
Processed 800/989
Processed 850/989
Processed 900/989
Processed 950/989


In [16]:
import numpy as np

def json_converter(obj):

    if isinstance(
        obj,
        (
            np.integer,
            np.int32,
            np.int64
        )
    ):
        return int(obj)

    if isinstance(
        obj,
        (
            np.floating,
            np.float32,
            np.float64
        )
    ):
        return float(obj)

    raise TypeError(
        f"Object of type "
        f"{type(obj)} "
        f"is not JSON serializable"
    )


with open(
        "vision_rag_output.json",
        "w"
) as f:

    json.dump(
        all_results,
        f,
        indent=4,
        default=json_converter
    )

print(
    f"Saved {len(all_results)} cases"
)

Saved 989 cases
